<div style="position:relative;overflow:hidden;padding:54px 42px 52px;border-radius:26px;background:radial-gradient(circle at 12% 10%,rgba(56,189,248,.26),transparent 30%),radial-gradient(circle at 90% 90%,rgba(245,158,11,.24),transparent 32%),linear-gradient(135deg,#061827 0%,#0c4a6e 44%,#0f766e 74%,#713f12 130%);color:white;border:1px solid rgba(125,211,252,.32)">
  <h1 style="font-size:44px;line-height:1.05;margin:0;letter-spacing:-.035em;color:#f8fafc">Kaggriculture Baseline</h1>
  <p style="margin:14px 0 0;font-size:19px;color:#bae6fd">Credited public schedule + a price-impact sell layer</p>
</div>

## What this is

A credited public action schedule, plus one narrow refinement: **the market queue is reordered so
the SELL with the most revenue at risk takes the earliest slot the schedule already used for
selling.** No order is added, removed, or resized; no purchase is displaced.

**Schedule source:** public episode `90203019`, seat 1 — team *Charismatic Pizza Party*,
submission `55273679`, final bank 183,721, played 2026-08-05. Public replays are observations of
actions that were played; nothing here claims knowledge of that entrant's implementation. Data via
the [Kaggriculture Episodes](https://www.kaggle.com/datasets/georgymamarin/kaggriculture-episodes)
dataset.

## Why slot order is worth anything

From the competition's Market Mechanics: orders are processed **concurrently across players, one
unit at a time** — both players' first order transacts, the market moves, then the second, and so
on. An order in an earlier slot therefore transacts *before* the opponent's volume of that product
has pushed the price down. Slot position is priced.

So the revenue actually at risk in a late slot is `n x (price(x) - price(x+n))` — the product's
price **steepness**, not its headline value. Using the engine's own curve
(`price(inv) = base - amp*f(inv-I0)`, `amp = target*base/f(T)`): strawberry and milk decay
linearly, melon and wool quadratically, while wheat and egg use log decay and barely move. The
steep goods are the ones a late slot bleeds.

## What changed, measured

Paired against the unmodified schedule — same seeds, same opponents, both seats, 432 games:

| agent | record | vs `closer_cleo` | every other opponent |
|---|---|---|---|
| unmodified schedule | 99 / 108 | 3 / 12 | 12 / 12 |
| **+ price-impact ordering** | **104 / 108** | **8 / 12** | 12 / 12 |

The entire gain is against `closer_cleo`, the strongest agent in the panel, with **no regression
anywhere**. An earlier version of this refinement ranked sells by total value instead of price
impact and scored strictly worse — the key was chosen by measurement, not argument.

In [ ]:
import base64, hashlib, importlib.metadata, subprocess, sys, tarfile, warnings, zlib
from pathlib import Path
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")
RAIN, SOIL, NIGHT = "#0891b2", "#8a5a2b", "#172554"

REQUIRED = (1, 32, 2)
try:
    installed = tuple(int(x) for x in importlib.metadata.version("kaggle-environments").split(".")[:3])
except Exception:
    installed = (0, 0, 0)
if installed < REQUIRED:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "kaggle-environments==1.32.2"], check=True)
import kaggle_environments
from kaggle_environments import make
assert tuple(int(x) for x in kaggle_environments.__version__.split(".")[:3]) >= REQUIRED
print("official simulator", kaggle_environments.__version__)

## 1. Write the agent

Stored zlib-compressed and base85-encoded; the emitted `main.py` is SHA-256 pinned and compiled
before writing.

Worth recording because it silently breaks wrapper-style refinements: `kaggle_environments`
resolves an agent file with `get_last_callable`, which takes the **last callable in module
namespace order**. Rebinding an existing `agent` updates its dict entry *in place* and keeps its
original position, so a helper defined afterwards becomes the last callable and is invoked as the
agent instead. Hence the `del agent` before rebinding — the cell below prints the resolved entry
point so this is visible rather than assumed.

In [ ]:
AGENT_B85 = "c-oD7Y1gVqvhMf#6}nX}=(ZX}L60K{0*a!DW3LvGK~W|dRGfbM#rNG+T|L~hZhjC+o`{Tyj456C<Hrxm3_{bgbS<T*q3JrQQ1SHNQ+f?9!6=N>0%V6&(bJC~KYlZ9&-KI9P6>3Z{4;e7T<2%%ZkfB^WU^XGHB&#-qJL@#tu^8*kD+64$Oq?1JvvQ5$~D%RHS-E98?HQ}*N?@f3g@dWcB5WT;ALfRqpBfpb4_(ptRB=wdSo9@9os5I(N+`}@k5?0oQFKy;4||fnC7GJ^tavxGm5_Ax?6-;l``%=l<zUOx;}>|vhf^bz}7v61@$>>Zh8h8FT445E^g*t&0{;ZJC3L^E4|%b-&n96P;%>_%#m0fX`X&Um371oN(8Xk?3&QJSejFh5qd-mO1G0G&^QlUdyYE9?afwcDxh%Ep~(_@j4`{V!ptPQ-EO>~If$zI!;+DiOPXg^o>LWcT`UXg1q>g`_{6p`sIqt*d&;;tjinCskXpNC_*kX2S8={vC{w0|7d3R}o+g4L@7VKthx*y_xMR=wlWlP&?1ee=EC`U-^d9n+t>R8rb4h8UD|Xu~TK=P_v6WB+I4biT8&yc0&o}BSyW*oVMHK4I=W%syqJg=boAF)O>dkoW<8-&^(!KJKyaRl?u?VWAojZYb{?^7fac=Ph=#a3vi$1XHLFLj{U0^4w@wT}vVV84<d8#o!S;CtZ;LojLjpp{<b~Jau;b2Ftr`-nkT1x(AU#W4gd100bk+rMsAD6YahBhyxgSyvxJ^NU_3Uh`p)~M;Z>yBx}Si2`|ioj~77t(aFIB5EMDD%hRF?`-`*ksCK*aSQ~1n2ddhlkqoj!5C21+Sn^!yB?FZs*taYOg)}>v|QX1q|DQZ6MxZkx^_kdy|%y#cqY1+H$6(i|aAFxW?E1qSV~<L4PhrQ%xu$-1Nmj7q;v_f_!KH^5{$CAL(LT%!m6b)Sk5k8A*?hA!trHCymHV3%iZdtx~$`@P+d@-Mkm6c`#5a@^FNm7bF$p!|8y0C!0F&AjRfM&GGx%R(h%h@jSW<k{iudz&e-C<JS9JH!phPljXunp*y$@(T>gx@Yd63xjs-F-?nmof<=0@vmHF7aiOs;)=0TupRe)aGQF~wobwJc#<1gR8A{%*wEd*Lr(Z<gLweV8Nb>Aa9rYx20_f01#Ik60w9YK^aIY(bE(-2744_mxJzw{Bm%{O!cec_U^&i&bFr3XaW-s7WUb>Aq)~#aM!nEkPk2WafR?h;~*3iu~Ski8tx&|CnkOf9-(>KvHyT`FSfrSp0C%}eSDkD2{JR}}_F^vd0d~aF8P*6DQJk#U$GHtm4zN`x~y4T1DYq@Z!`QwtNb@7-*x$R*l>5Fz$&sTeS#kN;C#!)ok%UW1_VrZ*W%6Iu<pSi{UiOR)`DQL^lZA{xfLvE2L-7AXW3MU?EBf=uN7?$;Hm3XfF=KA_N4Jo9mzpClgqusqTP&e0M<=yzuZ@a>Lmo2@)cT;i=f$LdHrk<lauyO8|$-(W>?P(>_&3C0&7u9Bi^GHz}961>j?&ty&i?dAw=K0~OdF84vX%Xd@#fG4x`$z9WvR&FNUcL3sQHBHmvJkg@&-U`iij3b>Q1oc)u%JiXopfKW!bm$XKDrIpF40=^{q$4J+~3*qIh;Q!bC-n?8G3Jd(dou_=6!{adZ%bt4G;l9ChI?ho~KLPF~~f`U`UsC)`Wr9<6dJ6R0rirX;|ZNHTU+i?J5Aj+szkKLxp#7;}7uKA};R825-NJ{RqB6NSD<huj!UZ4u-S9`o=euyo8vyL)D9r(3FS#X11?xMj+M+S4?2<gV9jAw5R%nLTYS-pA2T3Kmpo%lF-WZ9Ey9jt{_&P$Dkdb(DSo)aSIh>PPK{kbV*^GYlp=8QF-M|`{Zg5<TIRoAx1HBa5AS8f2}cBJHHuj-E$OX;DP0y%Mhu!0Xs=Q<65>ERJj~GymeY|=h!nJ>2&seXgJYI2n#P@L+JU%F#}E+KC8=6{va9YOgqcwL}tFoD8uP_E~&2F4(0s7Jq@bZ(4M4MxT^W)tWFTG*Qw6sIX#?P1iTX;L-l&>lI<I+y@f{9+Ukzc-3+<mymjl*$xfi+n`xWVmU`FM+=4EMUew{COYpwb8_!7Uy=e(GUJJCipX74rtPki5r;hODR=mTN<B2ZRI4bY(zTsm1;ngLrE192y>E^9Csq9~i+Ma*Q=dho#TYx(Q<t6&gxE3WrJ=Dggx2`r;{hADItE=gIyd2CGJZ6?*aaigr)(mf5!Aqxq&fl|3s{l{q@d;@)&5T`&u6GgzGoRdINSSNX%&NWPo4iH-UCU%CV)f{E>j1giKyC403>2aUoKcW1cdp8;WpFoY8lgF~i)^$5&ubcMb5XjOCGTy}bv!aK7lV6HIdn&@)c~yyA^Oy^1HOMgln0IMdV$>Q^~-EvN!df!o}4J=rocDa7=dNEWx=}#hBK_6Wde7}k_R|<nBFG72))8$V}V0!ZhDcuE`e8}sQ!{e4{o08cXeP)<S113gizP}g}W$fPo#-w?W8d0!u7=guj3hG7nLot47!+Ae$J%nie_vkcRD^?)!n{NLxOBT$5XBT%6PVaJ9f@H*1d!6*QPycWOa=T7AjGxvbAn?<iu$2zOVEoI-Hc}#3AkAPdhry%eg?RJqGtCyt%@xB+b)hDK5=e>ILP)Rei--Hlc4kvg8i2?Um3<Z7e(OZrcwp@JqFZdCa*Y5Q<4{yT?kG%-{`Qxn&kfXfL}Vp+}ASb4i5`Wlv?`2A(k97#h!2tGSnZ#{#ArEII>cZ007j>6^dp9K@uS&@*qxHrv4b*-Wo=zv9t$RLFTKSaK9}x+y%vt5y~p)T;HmvgQ1BIwYZ18y2cSdT=@+MZ9<RRO9v-_T)>@J%J^v7fovAl5t{3rXCsycU-g;cnz7sJBG%C)1i{i_GR;JR=BVgEzYQRZ!Rn-1U6wG8wR~O$SqlyDK%G$=bt!PXNL=t8#Tj?8&P=|B5IY^DNI7+hL6TD+#LsAr(L;QfrHgr?BzYwgchQ9r#)cUoo#v#adbC1Y87B;ahJW;8yyIkWaf>`Ld5}dcghThRBSRH+o@uTdKn1Qy9)L?@9`MSH_GSzzEx~?jLQgrW&5aT?Q&1-{gmC#ij>l27)dIeok7yW#@_7)wfEHpC&8B50V7s0>MyKwWnL4c%?S%~fQZ|cz2^a7e|$78_C7S^J$#ezm-Bhq!L^<);npb4ZwdK`JXXbgFeobw5SnODnoU+aKP0d3H@aDt`i-YhR0z{ht9aWgdc%VNOn`x?*hk{U><c%%VFouKi0y^ZgaWZGS%_?OR;46Sd&pr`umcl$J%IgyoK%nvv8rwdn+#KL5AH-wQ092xT}>O`B5QSgP0tSV(5KwRaeIl`8g(LT=mhiX85@Ndd-qu8FeX1?m1!<%35NAkUhq!&x;bz61EdSix0n_4E!|nE#EYJcHnMh`?E=sXV5GIj9$iX_(UsR^8E`sBmey=G*<KYA6&Hp(d++9lvdj*{)-7!ZU^BhzjKVJaxNLX5B3oLL+ukDYOe*zd;pINFCGT+S#I0T4)QNCE=^q9csSDk`I6b8&HtfqRS6LWVuHD*a5604(Uy?{ZMSi^kY|N&pv)E+<kXUWG(X@P4)M~A&+g?vbryI0pJZw=q@^PWHz0cAQ?zVfaJOxFBlK!4MQ^SV2C|kJ{sL9NE-q31=Bq7q>G><{!PA>`p4Z*@|cV|u{RbSL+p(>+JrLgNBO}s#t3j|OtUO>1IF??_Rd|fN-Hg6Jj{@S`mi()t99rEYn(qI(IzYq6>vBsemCzp=~PHzdPr!3n`!VrwgRp`p;YhlDQK((RZW)E3pYAoS4ab}BlCTTd52k)+a(4$BCvE#0wla4Z0E$)xbfLn`kkc>9kk=h<ITc_V$pKAMT{x(><4IB(q^g#>a@><!2phi4!9JT6e-CH!*)ehKhYb&u^jk?<-Jh@~VX6^B8U3$Q}J<i;FS-}a0HqFk5Zni4C9{iLqLEesA7lnMZxt)$8`iOJ!vKMvl^+T`M5sV{KJ-E*kEG*jNYvK7G>;Ni6acI%0K1ZWI9-e69oXs949Wn>Mf%qbN!%nW+jLhjA$JUD-gb)u9BqlS8-|~-q(ri2y-eq<+(x--YMwg6H!A=Bg);d|F_!>>~8c>v45BCg{&_pa7$I{uU)120{Zhbhmg`++$9M)7tS!TvTqsH)0Lwf_rxt<1vCvW;7yoL$%#4UEz;x04eBk8UOO9slf7LLp*&N@>7b_=|8C>G41B0eweV1CUE?z7=>kzrj@*%-@Fbv^Ngurls6ka`;)9S8e%h+m$o+!884&CRIwewqvNgf2$!XOdW9bhsZ#!$zkrJTy})N71=P?@45l>)8BC9?W_SL><e6L&Fbx(_ufInXR{k(k*Ml>Hs)J&)6MV4CbFrpRo~zL<oW(k=JfjvFLKnUb4cZBKc+Rd;tJ|J(k;WZZ3HZ2iQ1eEIdq%;q6Rw+lTC(L9*K1jw19J-<~}R?aFGj)5q>atpl@V_Uf0kEf-rlP|6_d3A39@Y@wchNjv6MnsgdDK`INK;#R*gzVS@&=UJ;ZE$6aAdcoW2Q5-DBq{?mH7AFtbIeUpP8=FH#2Vbha<m6i5E+!XZQM9xjNcNm!j>22ZoVd+)(0BSpA?qBvt8SVH`?fR<?=UCbMqtmlQu*k6XmM4`oHsY)EEbO8NYylmU1*JANKbYvB`+KtMY!>id+ztos5R`_^IOS;OBtq6*wX1^dbhMW!pdYbz1ox*G63!H+0xQca0pzMVPeS_EKFa>;f^`f;kul~c0+sG<=TqRR61&r0iYTjSZK9RN_-DfYWhH|UeVWLh?krBM|=S<%B0GQOL4ZHE4{RJ@y$)CR;N6-_PDG11hLAa;pkY+Kpl3|?@Vr3ZBf8W#m=EnEHPYlfTQt1Yst6#i#VU^P}E<Ma)1Ci?9~?73KeabfFwc0S&vy<CJztuR!l>k&(8+*HD{j<HoB7iQgL&1p=Ymf>4azlCbmS<3b4&Pa0-gx=0Kze<Y-(db>H~vBE^%rW_jatQ9kV!<Ln99N6Z4pXUeTb`6`K(7F_LC5B)u)tfjp|>`6=W8E(U<;)Hs!{6Zw$4N*}jd`of#ynoEMK?P=gwDP)m)7ti|Hi9EEJg%tm^3YIA7pt^w0iE5cC~T|g!ieTO3MNXDY1Zho_po_{wd+pCEM4z~3rKb30aoMBt;aPCOV;T=)$zi;L-~+i44o?ULWE0RV`&3B#1T7uqODi;KE6p^8MA73>ENvL^rMQ<*($uOdugD&a-Dhk$$O8Xzl@~z&{}6Nd%2oq)W9nhc8%M(UA*}{tdNMsqX1r+*bjue9O5H2;`5_Ee`sEBi3(%84z@IKiB;`17;-zZTcrm9Qtz(U!&$XCTMw$qVG1<w=?OnsLwX4D!n)|$X*cYjfVO3T&3R{qNFt+bmJZ%cYiGFJ>J&$2U+sByzEW(7BPtuGt#0#FAm`Vtb+aqw%fW(k8?xO2_d%X-A>3%|6dv^z-M`LN3|7>-s?4%?QPwUfwse~F(7h{Vxi%bTqf7?m6w4kSic=3k^=^rQl9<@nn!L%IU}uxdXAf@f+TTO&kP~SAI7O{MlA@c&v>b15?%MfMLCEE6W0~AMW~XWuU-VJUG3VB*eqyi2ZsVYQ!976l-b#F?gDzKogHOF2L3<%?`RN_tu+L-Op<#ccW@9*_4>m*A1%IcW@n%Vzh&*#D%?csEn6$D7t)$14pI?vA8S*AN;Jw#JpQ1ZhbZ(F?wOaP!{D$WH5-V}j({OK)8TEb6IQ;>x5e0?HDl>t_s|)bha>|AD6v+ki-sK#w#?=S#IE~cXifXPNZ|dF3r3c`4vECdn@K8T#EkfeiIzQ%S-7E0Y?;#LENnUN%*Y#|<FFjy#%SGFD8Y2Nbj)*ccIF?u2ysNb=2lC_EmEv}gDs*esQpp6OeXBZa$?vK8%$$(Q6QmD^;|w;=?|IBf6HEZOOzf3<s(>k#OnD=FH{8{RBnMrSHg=4V68u^cVx%Nzhj@AlC!G@BD=A%_zGu2>e&3&rg55k)@BJY<I;1zYH)(ETQUDjTIl<zcUZ{H$^m;E|QI+7>q@~yb3fOO)@9m=`9>eT%VG>JpjxtEjL$X0MxV9TQf4s)e6;|f=cJ2*Y*!fX;3&)1=id#%U7!wL)RPY+Z3L?5WpD5(IMWG9|tj?-rjxFyh-SlXTqqF-3&gzHNd(mDUIwnTyn^tu)Tp)4a?gI|mE$mM1>XZzsxJ}By^({3AgJLbJ9ap9K0L~Oh^?Dly(YyVO`AM$PH+Whtl<NA5?;kg1(%_J5d;Vtjx*z03gtc;mnUW)(-SMQ@tSMlraU9k1j83%tEtJXiYCs2_mL`$?)QxUV%~rq+)tnE~{)XR7u5n>!;%RlQiDiFuY=zV6QeBL0=W>-cvgheI$0@!iQEuN^&#)&{w$bKev?9-eI<j3$ZuHqh;bo2BjMk}C069bQKBKmG#u&)RrA~GQcrWkb&T7-=s$k$}*htPVlX{XmOmX+f?L;L~uG?}BkdgKdYP74)6q?*}#cZC{M(h0JEk<#<SB%H!JYEO-**!A}F<~?zv`aYHz6M$Gg?XJyP+scYe#e@3XR&3>8aGnek1v-8a{~A2py;n3MP`>5kkZbG(!{kdln)AjUh9v@pAOp11?5snNeBxYZrb;*X$#0#Y}G(+lg3pa3zycW?$$FCX4DJn%_rk*!(BhWP;{Zh@dXAlCz+_5ZK5?^ds=cWTF4ZBYp=2g%nuq=?YNu}6^S0U_z+jJ^~^RaX~=8dE*w3FBl}aVkqJkgJL|m=7AmX|cXVs>Q+o|2&AJza(gk~j3OOt7#McTzbL`vlnfTS6HH{mM<=XkpawIf41E4Og#SRCt_JLeEM|g9(lrG18>&PHfr`Ug{)1Dzt+QCdX3?SY!553W(>KplKmkkn!?B-v`ya<Vo(1oUp6|cpuGv0t?+pb)C{KlfIt@acy2iV0$4^?Rx7DTxRt>(4KagyF5xtFwmdDhwjFMGFCjHZS>YE;}D&+me@I`>o+u$vNFoYtE2Yt;eUMDX-(jK0W)#`|`p!@*=xk8T6Ldmx^W3dL>Cl&t<3NN0%WVjI*dkPM!C<$OQK<^lZD?I8r%GdiGR<p4lhgK3A~)MmZG20Eehcj0LwkqaR@H><H7j@6CXV}E#;xgzxJhe(AfoL6`aYx7EtPw)3!;zio7c2+6z^xPjYust7*M$b8HXxYW2eQO~QpCvQC8Nn?rI^MFUo1dE+O}EUPl&V$LmlSF)7h{TVoZ|Lg5^D!cxtanojIkA)Yry+_E-lv|0yPE)&$?b14*Vt`Pu3S?S>Fql8dGqN`|Ngf@Yu=rn5%*Nr8)t6V0(JNx#et8b;VL`eKgKzrEtoI9n#_ifXLA3gMj7RemJJH+u})X(T{CZp;}tDuN(SNCS=+?D2-ZqG%kZEDxVDXeFN4jdRb}m(_UY_>0KHd1+7kNg`pIB&E34*o8cutw|W>INV^2i_0_<)%P^D=K&R$y%Ncu|yBRy1yXTgT{h*P)x@`3l2u*aZl<L&To5C!+;q)s-T|0F~-@y$@?<3;X?iE7F0wv7SbFDH5YK?R-4o@P~4`{NF=loh`kTt8AINZ(C(qz`}%y}&A@y#Q@x#lW#mE;8G)&_E78W`z}pmQFw+~Jq4cMssrxM8(QRMM(4Lut@pJ2vRltvPn3OOFMYsb~C=c+%z40p;qq8=MYH{=?0sZF~<(Y7aZjJ-K&#19ii5z4w@hiCuM`bP*-tTDR1i6I=bwy7W>bj4o6V>1}1i*Sj%IPKm<EJ!f)QuX;~6yUtD`w`hh?yu2uk#P!RNjsAocRDL?KnRb?(@x>8V$!>AvDc;=(z1=J>xYQrck{{P*1=v~?(trRyBpf|#@96zzl+CHRY_Y<5COM_gFHL<MIJ6mVkd?amVWyVPobEw+p&if9{KV=-)XQF~^M<Bwy(xI$0Y6&J>MUA2Hpk_vcEJ%<Y!XFaqAYUy<<{l!@mQ~bin7>s=a(2CJl{2Tg2fV+95GpA7D#Qrc6J9#j9F|Nl+RV$<Mg1T8iUO|iN9<bY1>$~xzU($W%WkN`7<<rWBjqb3qWd_LA&12-MJ)4EYErCy`wi$_P&3X&VF3vwbgVs)XGG&o3z+9q@3{|Pj+RgKmeRFZL&FH^&T)Pnd#V!ZmL|4cSAg<H@ZDETptUlB{ANx<{{i&V4b9b-~!Z9hT&wi8t>ADtH*=&{C;A=<Pp<RWim16q_w5>IQy1}>8kp6R&&!}Z$6T)zp;u%;}ws&xV!w>Rs5<x)Yw*bq70q#xvkR=6x<W`UVD^;+3P)rwzHKB)NMyA4Q}&o09J&)F=$M(xg|wQGk8ax51N@9#_c1@H7B+Fxn4$8V0+VH&$vH;F*0;aldiuELGU<K-?@+1s*CQeE{SX(mrJhZ!Qrgs+~23yT8&Q?$m^mKV1@hg^{K}Uuo=`?RdEe*=Ut~l@r~#A+^)7bP86=RKk?=W%2EsnwhTj<%H0+9jIL*NnwFcVRxb;PPg{_vSe>bKs9u_J*#nt55UCv|yUT@@7uKoIj;E3ZR2^yJQ?>{T$2OOiP{P}hX422rozb;-?v&=Q_^N7|%f$rNNcLru=W$%<?jI1o%d{KW^yZ`v;<!?n*<x;!$F3WWfDW}Xre9C0Ts;p(j`X?@$~7Kxx5GCZHPGrJ*YwD_bgwIGJTpRWx+}~JWn<M(&Z3^S=mPs0oy`ozzGt`RSB@p6L37r9veV2_IM16vIDV}gt6UAaLDQqSH4QiIb~;9Xxsi!pkB9_p#Bs}9<%lt`vG?8fayVaB^5bKZh`Yr7FuQKvjpmR+%J`u&lkszdEt!4<A;-a)&K8aaqTCAl_A+<-t28t=rw<rOQkA%&;HWPt$7Un)%_ZJRuO_Ry+UmwQ2a!;@N{-cR2{~jY<mieG)2D2mj6k1DGsj@mL$gQxE@M5ZL;&GteaCJy1K6~PjD*xO`*Q7GpEt%hLZyMIF88EySkzdD^IWqPwKmv+tcZ)2FfE(abW7Z81Irzqv)qtplRfRi-DagpT%q0>V0*JySv#XM_OWj)THC_IZj_cU??shGVyW@Mi@cJ;O&HloGomUD2EnDV%HP6+R8XVLRpTi-ujRBnpVs86)#|c+e|t}k<0IKT3gv13P)UwXi-E^ulpHgUB@;|+AvrCzq&x$xLk2U{>>ipKHm9=9EjPTt)!~JonFpqFrl0Dts+s|`9N@EslUodCo5%PS#?Q=Cy4oj;hBMV^zJm6XLsAoc)aHQY-9___T`*>R+weL8u;idJ5aly46zB>vKt!SXe3J)nq`7ma)`)7@fl+;W-MiPt7Li@3M7vRBYVY-P<`@WRDCNG7o|}vll6)b1XTWUTB9@ACDIZT~CQi><x#~@RLem?AFeknW4I?c*ROePV%2rC9TDN{hd~aC_fsGn$#+Pe(b$zo-;uQtOL!-Q+Ah8-T&Sbm*L{C^0x@mTe=BM`}er#~cpl%u2^?f(T`a#pBx`WpQQPaJe^=KC+kC+aJcbT1rM4@er;Yq%8yY`CcbKi?cO3og?mgmf&yAijl3DgL2TpS52wz#z}x7`~$J6o7?$QFJ*UWefB)OWSqfXbcS{Bd+}R3Uw!TE$HP@JmC;x!L@(kJqhmQPNRu?1)pn9mhlgwyp7Yw#d3>ULXltBur}cvZqvLAaC1s{KiiXGJRg2klIn?X_y&d4dk_-=)pF6g_)^VJdrt~9tic>*6VksLpe(vNqCr9W^Mj@pJ?)-d%8|rExOe7sS|YW4m0<}fXns)njNM5+%--nTcrbdLc~CYEkV7CQV1%y@p4+s%@M1xzvIkwH$f{q!F^4`)vVA~k}l`&PNT6>S#;ZM_dHmey%KsqZ|Cb;K;JNH5LZ>{kSj)2VaRUOYB54|fC=ROmO7>-7Qm=O&rz-~pr9KoVh^<|$SD7UPI*w1#QG>i`rvGuIlrm{`(2v6N;k6?R-b`*p6#l54p6!J#C_W*CGu~&*9>xZ4<>S#*W+H3HrjT_R7=wyO9YxfaidDxKX(x-bQuB&CY2}AXAl$&R@3>bW~|d$xZf8#b-8%P2J4nP;p$I%lwHI(zd6m{vhkIjS4Ubd*x%v$rgTeJ&b%Ff!?dKR)KWxIDvW3IssLLwqaf4hG#)vdF`3sY)kn@RG}u;|ydmrK1t|H|Co{cTr3?iZXVfMhH1g&`E4n@#6=u15dJxh;6<3dz`t)+TNNd;2^mT8EYiw}JglaK5Gy#%6wX#w)9%q=^cm$JS_t6a5>0(wvomGE8@7Zg0qZ=%>t7g<Ew|hl{+aBj@1-^yPWVFD=EAYxv5_5d;LzWs$SCiTG0dCiALnR?Ojb}o$QBi|7#?EI@Ee|d`{9LWlmz6kJ^-6Hwu)V>qw-_E}Oz^FEA0zDo-e})@Xt%RG>O?f=>egZWZY)&_)_WUa=sPUiSVmH&a0oAAeIGuYNi4mr!w`lrpUU75>4Hqj0yzdqE2%y~tACMidYxL>*BcM#fP9jMNA03iU|WS~Kr<^QYh~I+a#$u|uz}A`x3jR7bB14j&eR@?=W3&ELD-iTC$q>cRVMqyrQr4E$2jMVGIK;?&8E4;rbYionk5v81#{~!>>cJ}MVaQf2y|DtXuTh6Bss0Ub#pS(VouyIAjRsp<snm8W=8eLLA0tztLJgMdKp_xx2;acJ+8C$oL7>R;`9_rJFGQuArqr%J{b0Iy{lux=U2r-mKIpRx8QXrcq?yd8{^VCfCkj^T_bBr{VDB7_3NOTe%yq6F`JEj-d&kdCWGcR8PH6p*);Ld_N+m(X`>~ulmHRqEm`F0`vt26JOo{KxSg+25_YjKQ+U^@(NUDJUW?JJ<CJt!?lZSWJX*Z{aksEp6TYg`nt5s+L95WaYRCI&RmDpA$rGc}>1Jh9oMb#08i91^BptY39ue1yU`Y+ezK@OU$Z2)@t467B#v5vm+|Z&oFc8t2q0Nk1m&&8#-080-d<KBo32^48yIMfJXV<x7%Kig6aSlbtyqD?KXW6=zuIF+#aDknq2{>u1fQq;I)+#s0tC``rl<VkDk)9<<smt#w*}GP(oX~p|!XYbY=7^VChu->p1UJ1tNWH7~o__|ma}3O%tNT7bI-koiL!Qvgayb$)cf;%lGVg7tP+{Q2@T;lJqH_Rs)&kra9Ti$R(61QNJ?<<RkC$;eFIq`SV$>CWM~#v+S}+OOE%5`}(3<VH48x5k=_~g&J*yL?$%W_3L`6ToO61sX7F2`6k7?}j)<ki3=#8OrPr6)|9NMduIKO02+1=%y*)x76_aJo8eB~xtxlj+g*N$4#Lb-eA(Dj@b3gHrtEp|Ys_t`u$mT-a^@aC~qqkG#~xEuD2C(>N!^fqnr0k>F?=QV$7u8y-kb0ni?i0_Z^cG5A(&J74;##f*br*lX)Y)$a-$!QJaalg=Mc@Ww<yd=BJFx*usm}N}tN|#Nu+A4_E!W_=nPz_^ev>EO8QbsmN^&Uvs6N+0_8`7rogS47D-q2U+QnMTx(8;^JN7e&h=?`8z4MqDx`3I@Kz>Od{_{_q4o0f{{Qv1|iyl;a#yoBq~MdVj6r*|z7yMxbb__ZL$%5}#f{4$<Bylr5rr5i(rFBRH(&vM<67Fm$ql<0tNA1ai8+$ohzn?j=TYaWdOKQUJ*ulBVEI=n)c&EY&=<Nak1JV(3qlj@-84czu51nKb0Lfy(;liqR2ZMxM)aMVCo#P|%D)Y9?Zv(edg`Jj;G^y^VAQ<j?bZJf?-tZw#r{_&ecjOkSK>+Ra#EmzS3@bil8@0zZ<w&&|X0DixK`}?k1`Alj$_<8*nf`0#X2SLBdJ%Z;+qB}?!J$@L9Z|nY#|4IG*$C#K-fBfTT>c>HGwBR%Kxv>@hq=#R3uit+AP1B8(ve%suba%nel<GQ$xo?hKM~D9Bw^TBMQ1=qf<~QQ^y&i%;K65{QreG-L`pM@P&NNbRs?q$$H&T}FfFJFCarq1-Ve^^#`v>xmKc^EyGccVXR2)?Ym7giiRKvfJeTJ`xk?*9G-~Mv*r{0McnN%?u$rHnLbUV=|bt$3B|D>ig&nL64ujzhj7pGjqFjZ5rQe%<p(kX5@Cc4rW?O#X2PWVs+snA!PKvBa~V7Xy&7Z*R*kCD{K_mgF;I5D2`5k3_j^pxv)iQ$~^{{*RXq(}PCRNy8;XHu3DCIWmB2fAeiDaH6q{2R^1wIW+j9hBs`<?D(Tr)>A4YhO6eH&y-jlyuM&e!hOuok;(@#vLivM~X1zo53lu<OiJC!S(f&lgg*SFFNoBrHZLvUkf&r+!a5-gq7#JTBLrK|9YI7N;Ewl(bMVAl#_^ShC%9(h+-v%O<fc#N@Q3h6XD<V{XP{1ddhP5sl*$VL<mPq8158bx4tE>C*I&Eaj5%#{5RXQPTvFxE5-kyQ?_oo&fl)CYo&s7ln{PdED<bD?JQmYurrh({5{okeLJ$0FQUZ!A7@DfO9;MfqdWVAXQG08byDDv5C^m*j|q_&CR)hBLGeD`ocVI4ZzGt_1xnoU6J3d66x&NoWqf9nJ1+6-#9McX0?GGZWFN?{(focUhl-y>Lqhn)<!=eS1SM$km%F)6Vuh<29!%$J=Y#KUR6c(5=l&-4&{D}}eYu6{?0^5`#d})`52?fvWc#<@<l*=e+rOoLELfV5lGyzkV5#ymRfgdtsFOWN;xQTU?<E-enIOA_C<=c-LG**He2~G}55J)B@Ax-T5{85%{DH(0PznAwB=(J*N+Pi!X;EAyXb>qSDCB$juN_Um$RE%l%?qQy2%4bKUtGThMgKt34D$yrL4E<z(tlNH$nk?ep&;^+6e*Sd>+%ah;QtBz(g%Sn|FZlqrafAeIG$V6;+IbFpX&d~6#l*Zfxdo|IXKyd1Omgq66d#Tfkx!0KU23~IqQ~yY~=>Rzb9;rB&&x>^!=G5KiE{W`6vH>=DA;K?e-tYMCSiQC2sfMkja+)2UfO!|2#;f_)j4cBmXyiG7loZ!>K~*7tx=>{Zc&f(j@nQU!MLm^@*l$Ha~v+5dX+cf21TMar`7NeMd!LnO>0E=}A;*|DOL6pI4Ll{jWs$`{#csAEu!n|5Hda)a}%#%=q7{{~h@UR8LafFQfnbu2f%7O8@w-Pv0u&H&yq-6#aEEUFSdaPB!do>QnBLkoziipVj^}+b17-AC}Ii(82#t2A_QJGbQ~@eLKdlJkd;fzx|O(nyG(J9v|Tmy;%GCuTCZX68^t9Rbuo%MNFu;?`r&YEq-0T2sPjREccz^z<+hvWP6=2@V~n8C;Mf7RRbfbE~zBheRPhTq!|3w8$R{rPoX{wL8*WB5BY!XEPnk5DKR!lOzFR(p#R+V@3tiYTV@b~-=yEo$ghUxXVTDoSs=Om|LylTlRq~1)3AKAf|B+H{ol~`DUsu!-O!%`JYHV{B$scdNi}JHbu*xS`i3S`i9uXH{!jFmy@P)z_@4bQytR)<{N4Pj9p=xJo2-_U!6a(@q}BLCk-zp(|Jr7`KU3hxl8JXlo~0*y{;8}V8@N!a)l8vZ_wL`}e|+VMPapN;AAdDfBEZK^rT@)N->ps}=C?S1H~;Z1?nl;y{+nOYVV#m1m_&3^_H+&WBKny+>2cFi?472h-2eG!zWL5IKff9vKhT?!AL)O_=O3(^-z)!bqJ5Rzq$vOVm6j5T?xy$A>gO*u(7*fM-#_@o9N!dw@!-Gz;Q{~n3ewNl-~Myeq>O*u{1c0xph&{R`t|MW^AiSt4VvHnF9)4Rss"
EXPECTED_SHA256 = "e853b64d1839aeca33ec7c634bb1f02ae7ab22881c604c010593f4dfa19e15f9"

source_bytes = zlib.decompress(base64.b85decode(AGENT_B85))
assert hashlib.sha256(source_bytes).hexdigest() == EXPECTED_SHA256
AGENT_SOURCE = source_bytes.decode("utf-8")
compile(AGENT_SOURCE, "main.py", "exec")
MAIN_PATH = Path.cwd() / "main.py"
MAIN_PATH.write_text(AGENT_SOURCE, encoding="utf-8")

from kaggle_environments.agent import get_last_callable
print("main.py", len(source_bytes), "bytes |", EXPECTED_SHA256[:16] + "...")
print("resolved entry point:", getattr(get_last_callable(AGENT_SOURCE), "__name__", "?"))

## 2. Live execution proof

Loaded from disk by the official environment. Both seats must finish all 720 turns `DONE`.

In [ ]:
rows, representative = [], None
for side in (0, 1):
    agents = [str(MAIN_PATH), "starter"] if side == 0 else ["starter", str(MAIN_PATH)]
    env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": 20260807}, debug=False)
    env.run(agents)
    final = env.steps[-1]
    assert len(env.steps) == 720
    assert final[side].status == "DONE" and final[1 - side].status == "DONE"
    rows.append({"side": side, "candidate_bank": float(final[side].reward or 0),
                 "starter_bank": float(final[1 - side].reward or 0),
                 "margin": float(final[side].reward or 0) - float(final[1 - side].reward or 0),
                 "steps": len(env.steps), "status": final[side].status})
    if side == 0:
        representative = env
live = pd.DataFrame(rows)
assert (live.margin > 0).all()
live

In [ ]:
steps = representative.steps
hours = np.arange(len(steps))
mine = [float(s[0].observation.farms[0].money) for s in steps]
theirs = [float(s[0].observation.farms[1].money) for s in steps]
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(hours / 24, mine, color=RAIN, lw=2.6, label="this version")
ax.plot(hours / 24, theirs, color=SOIL, lw=1.8, label="starter")
ax.fill_between(hours / 24, theirs, mine, where=np.asarray(mine) >= np.asarray(theirs),
                color="#e0f2fe", alpha=.55)
ax.set(xlabel="day", ylabel="coins", title="Direct file execution in the official simulator")
ax.legend(frameon=False); sns.despine(); plt.tight_layout(); plt.show()

## 3. Paired audit

Local `kaggle-environments==1.32.2` measurements, not Kaggle scores.

In [ ]:
AUDIT = pd.DataFrame([
    ("closer_cleo", 8, 3), ("slotter_silas", 12, 12), ("ledger_lena", 12, 12),
    ("broker_bea", 12, 12), ("rancher_rita", 12, 12), ("melon_mateo", 12, 12),
    ("homestead_hana", 12, 12),
], columns=["opponent", "with_reorder", "unmodified"])
x = np.arange(len(AUDIT))
fig, ax = plt.subplots(figsize=(14, 5.4))
ax.bar(x - .2, AUDIT.with_reorder, .4, color=RAIN, label="with price-impact ordering")
ax.bar(x + .2, AUDIT.unmodified, .4, color="#94a3b8", label="unmodified schedule")
ax.set_xticks(x, AUDIT.opponent, rotation=15)
ax.set(ylabel="games won of 12", ylim=(0, 14), title="Paired audit - 12 games per opponent")
ax.legend(frameon=False, ncol=2); sns.despine(); plt.tight_layout(); plt.show()
print("totals - with reorder 104/108, unmodified 99/108")
AUDIT

## 4. Package

In [ ]:
BUNDLE_PATH = Path.cwd() / "submission.tar.gz"
with tarfile.open(BUNDLE_PATH, "w:gz") as bundle:
    bundle.add(MAIN_PATH, arcname="main.py")
with tarfile.open(BUNDLE_PATH, "r:gz") as bundle:
    names = bundle.getnames()
    handle = bundle.extractfile("main.py")
    payload = handle.read() if handle else b""
assert names == ["main.py"]
assert hashlib.sha256(payload).hexdigest() == EXPECTED_SHA256
{"bundle": BUNDLE_PATH.name, "members": names, "source_sha256": EXPECTED_SHA256,
 "source_bytes": len(payload), "live_sides_done": int((live.status == "DONE").sum())}

---

## Honest limits

**A replayed schedule is fixed.** It does not react to the board, so it cannot recover from
anything its source episode did not meet — a weed on a tile it wants, a shop-unlock order that
shifts demand, an opponent that drains the products it is holding. Separate experiments confirmed
the constraint sharply: reordering orders helps, but *withholding* sales or *trimming* purchases
both make it strictly worse, because the schedule's later steps depend on the cash and inventory
its earlier steps produce.

**Local margins are not Kaggle scores.** Kaggle rates agents by live head-to-head play against the
full field. A coin margin against a fixed panel is a tool for choosing between candidates, and this
notebook uses it only for that. It also does not transfer across time: an agent from this lineage
that scored 2736 on 2026-08-04 was re-submitted on 2026-08-07 and rated around 1230, because the
field itself moved.

The step this design cannot take is reading board state on every board rather than replaying one.